# Creative Writing

Replicates the creativity steering from **"Steering Large Language Models to Evaluate and Amplify Creativity"** ([arXiv:2412.06060](https://arxiv.org/abs/2412.06060)) on Llama-3-8B-Instruct, end to end in one engine:

1. **Construction** — contrastive pairs (`contrastive_pairs.json`) put a creative-writer persona on fantastical premises vs a factual-reporter persona on mundane ones; the difference of means of the last-token hidden states gives a creativity direction (`create.gguf`).
2. **Steering** — adding it at layers 16–29 on every token pushes a plain story prompt toward more fantastical, stylized writing.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "meta-llama/Meta-Llama-3-8B-Instruct")  # meta-llama/Meta-Llama-3-8B-Instruct

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)
tokenizer = llm.get_tokenizer()

## Vector construction

In [ ]:
import json

with open("contrastive_pairs.json", encoding="utf-8") as f:
    sample_pairs = json.load(f)


def make_prompt(system, user):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, return_dict=False, add_generation_prompt=True,
    )
    return {"prompt_token_ids": prompt_ids}


# Creative persona + fantastical premise vs. factual persona + mundane one.
formatted_positive = [
    make_prompt("You are a creative writer who loves unexpected and dramatic twists.", p["creative"])
    for p in sample_pairs
]
formatted_negative = [
    make_prompt("You are a factual reporter who writes about mundane, everyday events.", p["mundane"])
    for p in sample_pairs
]

In [ ]:
from easysteer.capture import capture_batches
from vllm.capture import SelectSpec

# Only the last prompt row feeds the extractor, so select it at the source.
batches = capture_batches(
    llm,
    formatted_positive + formatted_negative,
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Consume each batch once: mean(creative) minus mean(mundane).
labels = [True] * len(sample_pairs) + [False] * len(sample_pairs)
control_vector = extract(
    batches,
    labels,
    method="diffmean",
    token_pos=-1,
    normalize=True,
)
control_vector.export_gguf("create.gguf")


## Steering

In [ ]:
messages = [
    {"role": "user", "content": "Write a story about a town."},
]
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_dict=False, add_generation_prompt=True,
)
prompt = {"prompt_token_ids": prompt_ids}
params = SamplingParams(temperature=0, max_tokens=256, skip_special_tokens=False)

baseline = llm.generate(prompt, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
# Add the creativity direction at layers 16-29, on every prompt and
# generated token.
steering = SteeringSpec(vectors=[
    VectorSpec(
        source="create.gguf",
        scale=1.5,
        layers=list(range(16, 30)),
        apply=ApplySpec(prompt="all", generation="all"),
    ),
])

steered = llm.generate(prompt, params, steering=steering, use_tqdm=False)
print("=====Creative Steered=====")
print(steered[0].outputs[0].text)

The steered story carries a more mystical, stylized tone than the baseline.